# Receipt Auditor
A vision LLM reads receipt photos into structured data. We measure how accurate it is against a human-labelled dataset (CORD), then run simple checks on the extracted data.

**Pipeline:** photo → vision model → Pydantic-validated data → tables → checks (items add up, duplicates, spikes, monthly summary)

**Notebook map**
1. Setup
2. Load the dataset
3. Build the answer key (human-written totals, cleaned into numbers)
4. Accuracy: extractor vs answer key on 50 random receipts
5. Checks on the extracted data

Run the cells top to bottom. Sections 4 and 5 make AI calls and save progress to files, so a stopped run continues where it left off.

## 1. Setup
The Groq key is typed into a hidden prompt, so it is never saved inside the notebook.

In [ ]:
!pip install -q -r requirements-notebook.txt

In [ ]:
from getpass import getpass
from groq import Groq
import json, os, random, re, time
import pandas as pd

from receipt_extractor import Receipt, ask_receipt, to_number

client = Groq(api_key=getpass("Paste your Groq API key and press Enter: "))

def extract(pil_image):
    """Read one receipt photo. Returns a Receipt, or None if every attempt was invalid."""
    return ask_receipt(client, pil_image)

## 2. Load the dataset
CORD-v2: 1,000 receipt photos with human-written answers (800 train, 100 validation, 100 test). We never train on it. We only use it to *grade* the extractor, so any receipts are fair to test on.

In [ ]:
from datasets import load_dataset

ds = load_dataset("naver-clova-ix/cord-v2")
print(ds)

## 3. Build the answer key
**Idea:** the dataset stores the correct answer for each receipt as a JSON *string*. We read it into a table, then turn price text into numbers so we can compare with the AI later.

The data is messy, so two helpers handle it:
- `get_menu`: `menu` is a list, except on receipts with one item, where it is a single dict.
- `as_dict`: `sub_total` is one box, except on a few receipts where it is a *list* of boxes. We merge them into one (later entries win).

In [ ]:
def get_menu(gt_parse):
    menu = gt_parse.get("menu", [])
    return [menu] if isinstance(menu, dict) else menu

def as_dict(value):
    if isinstance(value, list):
        merged = {}
        for part in value:
            merged.update(part)         # later entries overwrite earlier ones
        return merged
    return value if value else {}

In [ ]:
rows = []
for split in ["train", "validation", "test"]:
    for r in ds[split]:
        gt = json.loads(r["ground_truth"])["gt_parse"]
        rows.append({
            "split": split,
            "n_items": len(get_menu(gt)),
            "subtotal": as_dict(gt.get("sub_total")).get("subtotal_price"),
            "total": as_dict(gt.get("total")).get("total_price"),
        })

truth_df = pd.DataFrame(rows)
truth_df.head()

**Price text to numbers.** Prices come in several styles: `1,591,600`, `48.000`, `70000.00` (with cents), `Rp 116000`. `to_number` (in `receipt_extractor.py`) drops separators, cents and currency prefixes. It returns nothing when a receipt lists two *different* totals, because we cannot tell which is right.

*A bug found during error analysis:* the first version kept every digit, so `70000.00` became 7,000,000. The AI's answer (70,000) looked wrong, but the answer key was wrong. Fixing this changed 50 totals.

In [ ]:
truth_df["total_num"] = truth_df["total"].apply(to_number)
truth_df["subtotal_num"] = truth_df["subtotal"].apply(to_number)

print(truth_df[["total_num", "subtotal_num"]].isna().sum(), "\n(missing or ambiguous values)")
print("largest total:", truth_df["total_num"].max())

## 4. Accuracy: extractor vs answer key
We test on 50 random receipts from the validation and test parts that have a usable correct total. `random.seed(42)` makes the pick repeatable.

In [ ]:
usable = truth_df[(truth_df.index >= 800) & truth_df["total_num"].notna()].index.tolist()
random.seed(42)
sample_ids = random.sample(usable, 50)

def get_image(row_id):
    """truth_df lists train rows first (0-799), then validation (800-899), then test (900-999)."""
    if row_id < 800:
        return ds["train"][row_id]["image"]
    if row_id < 900:
        return ds["validation"][row_id - 800]["image"]
    return ds["test"][row_id - 900]["image"]

Run the extractor on all 50. Results are saved after **every** receipt, so a stopped run (internet, Groq's free-tier limit) continues where it stopped. One failed receipt does not stop the loop.

To redo failed receipts, remove them from the file first (the loop skips anything already in it):
```python
r = pd.read_csv("stage3_results.csv"); r[r["error"].isna()].to_csv("stage3_results.csv", index=False)
```

In [ ]:
RESULTS_FILE = "stage3_results.csv"
results = pd.read_csv(RESULTS_FILE).to_dict("records") if os.path.exists(RESULTS_FILE) else []
done = {r["row_id"] for r in results}

for row_id in sample_ids:
    if row_id in done:
        continue
    try:
        receipt = extract(get_image(row_id))
        ai_total = receipt.total if receipt else None
        error = None if receipt else "invalid answer after 3 tries"
    except Exception as problem:
        ai_total, error = None, f"{type(problem).__name__}: {problem}"[:200]

    results.append({"row_id": row_id, "ai_total": ai_total, "error": error})
    pd.DataFrame(results).to_csv(RESULTS_FILE, index=False)
    print(f"{len(results)} / {len(sample_ids)} done", end="\r")
    time.sleep(2)

print("\nFinished:", len(results), "receipts")

**The accuracy number.** The correct total is looked up from the answer key *here*, every time, so a fix to the key is always picked up. A receipt where the AI gave no total counts as wrong.

In [ ]:
res = pd.DataFrame(results)[["row_id", "ai_total", "error"]]
res["correct_total"] = res["row_id"].map(truth_df["total_num"])
res["match"] = res["ai_total"] == res["correct_total"]
failed = res["error"].notna()

print("Receipts tested               :", len(res))
print("Total exactly correct         :", res["match"].sum(), f"({res['match'].mean():.0%})")
print("Calls that failed or invalid  :", failed.sum())
print("AI answered but said no total :", (res["ai_total"].isna() & ~failed).sum())
if (~failed).sum() > 0:
    print(f"Accuracy when the AI answered : {res.loc[~failed, 'match'].mean():.0%}")

**Mistakes.** A mismatch can be the AI's fault or the answer key's. Always look at the photo.

In [ ]:
wrong = res[~res["match"]].copy()
wrong["difference"] = wrong["ai_total"] - wrong["correct_total"]
wrong

In [ ]:
if len(wrong) > 0:
    display(get_image(int(wrong.iloc[0]["row_id"])).resize((400, 600)))
    print(wrong.iloc[0])
else:
    print("No mistakes to show")

## 5. Checks on the extracted data
**Idea:** the model can be wrong, so simple rules flag receipts for a human to look at. They flag, they do not decide.

First, extract the **full** receipt (items, subtotal, store, date) for 20 receipts. Saved after each one; delete `extracted_receipts.json` for a fresh run.

In [ ]:
EXTRACT_FILE = "extracted_receipts.json"
if os.path.exists(EXTRACT_FILE):
    with open(EXTRACT_FILE) as f:
        extracted = json.load(f)
else:
    extracted = {}

for row_id in sample_ids[:20]:
    if str(row_id) in extracted:
        continue
    try:
        receipt = extract(get_image(row_id))
    except Exception as problem:
        print("failed on", row_id, "->", type(problem).__name__)
        continue
    if receipt:
        extracted[str(row_id)] = receipt.model_dump()
        with open(EXTRACT_FILE, "w") as f:
            json.dump(extracted, f)
    time.sleep(2)

print(len(extracted), "receipts extracted")

**Two tables:** one row per receipt, one row per item.

In [ ]:
receipt_rows, item_rows = [], []
for rid, data in extracted.items():
    receipt_rows.append({
        "receipt_id": int(rid),
        "store": data["store_name"],
        "date": data["date"],
        "subtotal": data["subtotal"],
        "total": data["total"],
        "items_sum": sum(item["price"] for item in data["items"]),
        "n_items": len(data["items"]),
    })
    for item in data["items"]:
        item_rows.append({"receipt_id": int(rid), **item})

receipts = pd.DataFrame(receipt_rows)
items = pd.DataFrame(item_rows)
receipts.head()

### Check 1: do the items add up?
The item prices should sum to the subtotal. With no subtotal, items equal to the total is fine; otherwise we say "cannot check" (tax or service charge could explain the gap).

*Known limitation:* a receipt with an **item discount** is flagged falsely. In the sample, receipt 808 had items summing to 19,500 and a 7,800 discount, giving 11,700. The AI read everything correctly, but the schema has no discount field.

In [ ]:
def check_items(row):
    if pd.notna(row["subtotal"]):
        return "ok" if row["items_sum"] == row["subtotal"] else "MISMATCH with subtotal"
    if pd.notna(row["total"]) and row["items_sum"] == row["total"]:
        return "ok"
    return "cannot check (no subtotal)"

receipts["items_check"] = receipts.apply(check_items, axis=1)
print(receipts["items_check"].value_counts())
receipts[receipts["items_check"] != "ok"]

### Check 2: duplicate bills
Same store, date, total and item count means probably the same bill uploaded twice. `keep=False` marks both copies. The real receipts are all different, so we add one **deliberate copy** (id 9999) to prove the detector works.

In [ ]:
with_total = receipts.dropna(subset=["total"])
test_table = pd.concat([with_total, with_total.iloc[[0]].assign(receipt_id=9999)], ignore_index=True)
test_table[test_table.duplicated(subset=["store", "date", "total", "n_items"], keep=False)]

### Check 3: unusual spikes
The IQR rule: anything above `Q3 + 1.5 × (Q3 − Q1)` is unusually high.

*Limitation:* this compares every bill with one pool. A small bakery bill and a big restaurant bill are judged together, so big bills get flagged. Real use would compare within the same store or category, or over time.

In [ ]:
q1, q3 = receipts["total"].quantile([0.25, 0.75])
limit = q3 + 1.5 * (q3 - q1)
print("Totals above", int(limit), "are flagged as spikes")

receipts["spike"] = receipts["total"] > limit
receipts[receipts["spike"]]

### Monthly summary
Needs dates, and the AI only reads a date if one is printed on the receipt. `parse_date` reads `2026-01-05` year-first and `05/01/2026` day-first (the usual order in Indonesia). Unreadable dates become empty instead of crashing.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def parse_date(text):
    if text is None or pd.isna(text):
        return pd.NaT
    text = str(text).strip()
    year_first = text[:4].isdigit()
    return pd.to_datetime(text, errors="coerce", dayfirst=not year_first)

def monthly_summary(table):
    table = table.copy()
    table["date_parsed"] = table["date"].apply(parse_date)
    print("Receipts with a usable date:", table["date_parsed"].notna().sum(), "of", len(table))
    table = table.dropna(subset=["date_parsed", "total"])
    table["month"] = table["date_parsed"].dt.to_period("M")
    return table.groupby("month")["total"].agg(count="count", total_spent="sum")

monthly_summary(receipts)

The same function on a tiny **made-up demo table** (not real receipts), to show the output when dates exist:

In [ ]:
demo = pd.DataFrame({
    "date": ["2026-01-05", "2026-01-20", "2026-02-03", "2026-02-14", "2026-02-28", "2026-03-09"],
    "total": [120000, 45000, 80000, 60000, 300000, 95000],
})
monthly_summary(demo)